# THERMA SEARCH AI · 05 · WiSARD Zero-Shot External Evaluation

**Runtime → Run all / Çalışma zamanı → Tümünü çalıştır.** GPU önerilir; GPU yoksa değerlendirme CPU’da yapılabilir ve cihaz raporlanır. Eğitim veya fine-tuning yoktur. Google Drive ilk bağlantıda erişim izni ister. Hücre düzenleme veya eşik seçimi gerekmez.

Mevcut HIT-UAV modeli: `ThermaSearchAI/runs/hit_uav_yolo11s/run_yolo11s_20261120T151540Z_78fb2af9/best.pt`.
Mevcut veri: `ThermaSearchAI/datasets/WiSARD_Multi_Modal_Sample.zip`.

**Yalnızca IR.** VIS görüntüleri, IR/VIS eşleştirme ve zaman senkronizasyonu kullanılmaz. Model/veri indirilmez; mevcut checkpoint ve ZIP salt okunur kullanılır. Kod/çalışma zamanı paketlerinin kurulumu veri/model indirmesi değildir.

Önceki kullanıcı sayımları 264 IR / 1006 Person / 3 boş TXT yalnızca tarihsel referanstır; yeniden hesaplanır. MtErie tek uçuş ve zamansal bağımlı kareler içerir. Bu sınırlı çapraz veri seti stres testi, bağımsız kurtarma görevlerine genelleme kanıtı değildir. HIT-UAV ve WiSARD skorları doğrudan karşılaştırılabilir benchmark skorları olarak sunulmaz.

## 1 · Sabit kod ve çalışma zamanı
Mevcut değerlendirme prosedürüyle aynı `ultralytics==8.3.228` kullanılır. Notebook 01–04 ve frontend değiştirilmez. Bu hücre checkpoint yüklemez veya eğitim başlatmaz.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import json

REPOSITORY = 'https://github.com/hilalaltunayy/therma-search-ai.git'
CODE_REF = 'feature/wisard-zero-shot-evaluation'  # Teslimde sınanan yardımcı kod commit’ine sabitlenecek.
local_code_root = os.environ.get('THERMA_CODE_ROOT')  # Yalnızca yerel sentetik doğrulama.
if local_code_root:
    PROJECT_ROOT = Path(local_code_root).resolve()
else:
    PROJECT_ROOT = Path('/content/therma-search-ai-external-eval')
    if not PROJECT_ROOT.exists():
        subprocess.run(['git','clone','--quiet',REPOSITORY,str(PROJECT_ROOT)],check=True)
    if subprocess.check_output(['git','-C',str(PROJECT_ROOT),'remote','get-url','origin'],text=True).strip()!=REPOSITORY:
        raise RuntimeError('Mevcut kod klasörünün kaynağı beklenen repo değil.')
    if subprocess.check_output(['git','-C',str(PROJECT_ROOT),'status','--porcelain'],text=True).strip():
        raise RuntimeError('Yerel kod değişikliği var; üzerine yazılmadı. Temiz çalışma zamanı kullanın.')
    subprocess.run(['git','-C',str(PROJECT_ROOT),'fetch','--quiet','--depth','1','origin',CODE_REF],check=True)
    subprocess.run(['git','-C',str(PROJECT_ROOT),'checkout','--quiet','--detach','FETCH_HEAD'],check=True)
    subprocess.run([sys.executable,'-m','pip','install','--quiet','--upgrade-strategy','only-if-needed','-r',str(PROJECT_ROOT/'requirements-training-colab.txt')],check=True)
sys.path.insert(0,str(PROJECT_ROOT))
for module in list(sys.modules):
    if module=='src' or module.startswith('src.'):
        del sys.modules[module]
from src.evaluation.wisard_external import ExternalEvalConfig, run_wisard_external_evaluation
from IPython.display import display, Image as DisplayImage
GIT_COMMIT = subprocess.check_output(['git','-C',str(PROJECT_ROOT),'rev-parse','HEAD'],text=True).strip()
print('Git commit:',GIT_COMMIT)

## 2 · Drive ve mevcut girdiler
Kaynak dosyalar aranıp başka checkpoint ile değiştirilmez. Eksik girdi değerlendirmeyi durdurur; çalışma dizininde hata raporu oluşur. Çıktılar her çalıştırmada yeni ve benzersiz bir klasöre yazılır.

In [ ]:
if not local_code_root:
    from google.colab import drive
    drive.mount('/content/drive')
CHECKPOINT = Path(os.environ.get('THERMA_EXTERNAL_MODEL','/content/drive/MyDrive/ThermaSearchAI/runs/hit_uav_yolo11s/run_yolo11s_20261120T151540Z_78fb2af9/best.pt'))
ARCHIVE = Path(os.environ.get('THERMA_EXTERNAL_ZIP','/content/drive/MyDrive/ThermaSearchAI/datasets/WiSARD_Multi_Modal_Sample.zip'))
OUTPUT_ROOT = Path(os.environ.get('THERMA_EXTERNAL_OUTPUT','/content/drive/MyDrive/ThermaSearchAI/reports/wisard_external_eval'))
WORK_ROOT = Path(os.environ.get('THERMA_EXTERNAL_WORK','/content/therma-external-eval-work'))
CONFIG = ExternalEvalConfig()
print('Checkpoint mevcut:',CHECKPOINT.is_file(),CHECKPOINT)
print('ZIP mevcut:',ARCHIVE.is_file(),ARCHIVE)
print('Sabit protokol:',CONFIG)
print('Yeni rapor dizinleri:',OUTPUT_ROOT)

## 3 · Doğrulama ve donmuş model değerlendirmesi

Doğrulanmış WiSARD parser’ı ile IR 640×512 görüntüler, sınıf 0 Person, normalize YOLO kutuları ve dosya eşleşmeleri denetlenir. Boş TXT **kesin negatif değildir**; varsayılan olarak skor dışıdır ve inceleme contact sheet’i üretilir. Bozuk/eksik/geçersiz IR anotasyonu varsa skorlamanın tamamı tanılama hatasıyla durur.

Model/ZIP SHA256, Git, Python/PyTorch/Ultralytics ve cihaz bilgisi kaydedilir. Kaynak checkpoint’e yazılmaz. IR kopyaları yalnızca yeni yerel çalışma dizinindedir; önceki run, veri ve raporlar korunur.

İki ayrı ölçüm: Ultralytics mAP için conf tabanı=.001; özel TP/FP/FN için confidence=.25 ve eşleme IoU=.50. Her ikisinde imgsz=640, NMS IoU=.70, max_det=300. WiSARD sonucundan eşik optimize edilmez. Küçük nesne recall’u orijinal GT boyutlarında width<16, height<16 ve her iki kenar<16 piksel paydalarını kullanır; bütün görüntüdeki aynı bire bir eşleme paylaşılır.

In [ ]:
EXTERNAL_RUN_DIR, EXTERNAL_SUMMARY = run_wisard_external_evaluation(
    CHECKPOINT, ARCHIVE, OUTPUT_ROOT, WORK_ROOT,
    git_commit=GIT_COMMIT, config=CONFIG,
)

## 4 · Kaydedilen gerçek ölçümler ve şekiller
Aşağıdaki sonuçlar yalnızca mevcut checkpoint ve gerçek ZIP ile tamamlanan çalışmadan üretilir. Sentetik doğrulama metrikleri gerçek model başarımı değildir. GT kırmızı, eşleşen tahmin mavi, FP mor ve kaçırılan GT turuncu kesiklidir. Şekiller 200 DPI’dır.

In [ ]:
print('Durum:',EXTERNAL_SUMMARY['status'])
print('Yeni bağımsız girdi sayımları:',EXTERNAL_SUMMARY.get('coverage'))
print('Ultralytics mAP (ayrı prosedür):',json.dumps(EXTERNAL_SUMMARY['ultralytics_map'],indent=2))
print('Sabit eşik ve küçük nesneler:',json.dumps(EXTERNAL_SUMMARY['fixed_threshold'],indent=2))
for pattern in ['empty_annotation_review_*.png','fixed_metrics_and_small_recall.png',
                'confidence_and_localization.png','fixed_confusion_matrix.png',
                'map_curve_00_*.png','comparison_01.png','false_positive_01.png',
                'false_negative_01.png','small_person_01.png']:
    for path in sorted((EXTERNAL_RUN_DIR/'figures').glob(pattern)):
        display(DisplayImage(filename=str(path),width=1000))
print('Kaynak model/ZIP değişmedi:',EXTERNAL_SUMMARY['inputs_unchanged'])
print('Raporlar:',EXTERNAL_RUN_DIR)
print('Markdown rapor:',EXTERNAL_RUN_DIR/'evaluation_report.md')
print('Sınırlılık: tek uçuş ve ilişkili kareler; bağımsız görev genellemesi kanıtlanmadı.')

## Bilimsel okuma notu
FP/görüntü, belirsiz boş anotasyonlar dışarıda olduğu için yalnızca **skorlanan anotasyonlu IR alt kümesini** tanımlar; gerçek negatif sahnelerde yanlış alarm oranı değildir. Küçük nesne grupları örtüşür ve paydaları toplanmaz. TP IoU özeti yalnızca tespit edilen nesneleri kapsar. GPU/CPU ve kütüphane sürümleri sonuçları etkileyebilir. Bu çalışmanın skorları HIT-UAV test skorlarıyla doğrudan eşit benchmark ölçümleri değildir.

`evaluation_summary.json`, `evaluation_report.md`, `per_image_metrics.csv`, `exclusion_manifest.csv`, `predictions.jsonl`, `data_manifest.json`, `input_hashes.json`, `reproducibility.json`, eğri verileri, figür manifesti ve çıktı SHA256’ları otomatik kaydedilir. Boş veya hatalı kategori için sonuç/görsel uydurulmaz.